# ESM-2 Tutorial: Embeddings and Fine-Tuning on GFP

> New to ALF? Start with the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

**ESM-2** (Evolutionary Scale Modeling 2) is a protein language model from Meta AI, pre-trained on hundreds of millions of protein sequences. This notebook shows you how to:

1. Load ESM-2 as a frozen embedding extractor
2. Extract per-sequence embeddings for GFP (Green Fluorescent Protein) variants
3. Score sequences zero-shot with pseudo-log-likelihood (PLL)
4. Visualise the embedding space with UMAP, coloured by fluorescence
5. Train a regression head on top of frozen embeddings
6. Use `predict()` on the trained head
7. Fine-tune the full backbone with masked language modelling (MLM)

`ESM2TrainConfig` selects the behaviour via its `mode` field:
- `mode="linear_head"` (default) — freeze the backbone, train a linear head for regression or classification
- `mode="likelihoods"`, `freeze_backbone=True` — zero-shot PLL scoring, no training
- `mode="likelihoods"`, `freeze_backbone=False`, `loss_fn="mlm"` — MLM fine-tuning of the full backbone

**Expected runtime:** < 10 minutes on CPU using `facebook/esm2_t6_8M_UR50D` (8M parameters, ~31 MB download on first run). Most of the time is spent on zero-shot PLL scoring (Sections 3b and 7), which masks one residue at a time — cost scales with sequence length, so only a few sequences are scored.

## Setup

These tutorials are written for **dev mode** — running from a local clone of the ALF repository.
ESM-2 needs the `transformers` package, which ALF exposes through the optional `esm2` extra
(mirrored as the `esm2` dependency group in `tutorials/pyproject.toml`). From the `tutorials/`
directory, sync that group so the extra is installed alongside the tutorial dependencies:

```bash
uv sync --group esm2   # installs alf_core, alf_tools[esm2] and tutorial deps (CPU PyTorch by default)
```

Register the environment as a Jupyter kernel, then select the `alf` kernel in this notebook:

```bash
uv run ipython kernel install --user --env VIRTUAL_ENV "$(pwd)/.venv" --name=alf
```

`uv sync` installs the **CPU** build of PyTorch by default. For GPU acceleration, see the
[GPU support section of the Installation Guide](https://instadeepai.github.io/alf/installation.html#gpu-support).

**Not running from a clone?** If you installed ALF with `pip`, run the optional cell below to install
this tutorial's dependencies into the current kernel.

In [ ]:
# Optional — only needed if you are NOT running from a cloned repo via `uv sync`.
# Installs ALF and this tutorial's dependencies into the current kernel, then restart the kernel.
# %pip install "alf_tools[esm2]" umap-learn matplotlib pandas

The cell below auto-selects the device (`"cuda"` if a GPU is visible to PyTorch, otherwise
`"cpu"`). To actually run on GPU you need two things: a **CUDA build of PyTorch** (the default
`uv sync` installs the CPU build — see the
[GPU support section of the Installation Guide](https://instadeepai.github.io/alf/installation.html#gpu-support)),
**and** the Jupyter kernel must point at the environment that has it. On CPU everything still works;
it is just slower.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import umap
from alf_core import BaseDatasetConfig, LabelledCandidates, Modality, ProblemType
from alf_tools.datasets.gfp import GFP
from alf_tools.models.esm2 import ESM2Model, ESM2ModelConfig, ESM2TrainConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load the GFP Dataset

**Green Fluorescent Protein (GFP)** is a classic protein engineering benchmark. The dataset contains ~1 000 nucleotide sequences encoding GFP variants, each labelled with `medianBrightness` — a proxy for how well the variant fluoresces.

`GFP.load_dataset()` downloads and caches the CSV on first call, then returns a `LabelledCandidates` object where each `Candidate.data` is a nucleotide sequence string and `.labels` is a `numpy` array of brightness scores.

We use 50 sequences for embedding and fine-tuning to keep CPU runtime under 5 minutes.

> **Note on sequence type:** ESM-2 is a protein language model trained on amino acid sequences. The GFP dataset provides nucleotide (DNA) sequences, but since the characters A/C/G/T are valid amino acid single-letter codes (Ala/Cys/Gly/Thr), the tokenizer accepts them. This tutorial focuses on demonstrating the ALF API; in production use you would translate codons to amino acid sequences before embedding with ESM-2.

In [ ]:
gfp_config = BaseDatasetConfig(
    name="gfp",
    modality=Modality.SEQUENCE,
    problem_type=ProblemType.REGRESSION,
    seed=42,
    train_ratio=0.8,
    test_ratio=0.1,
    validation_frac=0.1,
)
gfp = GFP(gfp_config)
data: LabelledCandidates = gfp.load_dataset()

print(f"Total sequences : {len(data.candidates)}")
print(f"Example sequence: {data.candidates[0].data[:40]}...")
print(f"Brightness range: {data.labels.min():.2f} – {data.labels.max():.2f}")

We trim the dataset down to a small working set. The first `n_max` candidates (and their matching labels) are reused for both the embedding/zero-shot sections and the supervised/MLM sections below. Keeping `n_max` small is what holds the CPU runtime in check; bump it up if you have a GPU to hand.

In [ ]:
n_max = 50

embed_candidates = data.candidates[:n_max]
embed_labels = data.labels[:n_max]  # numpy array, shape (n_max,)

finetune_candidates = data.candidates[:n_max]
finetune_labels = data.labels[:n_max]  # numpy array, shape (n_max,)

## 2. Initialise ESM-2 as a Frozen Embedding Extractor

**`ESM2ModelConfig`** controls the architecture:
- `model_id` — any `facebook/esm2_*` HuggingFace checkpoint
- `pooling` — how to collapse per-token hidden states to one vector: `"mean"` (average over all non-padding positions, CLS and EOS included), `"cls"` (first token), or `"last_hidden_state"` (full sequence tensor)
- `repr_layer` — which transformer layer to read; `-1` is the final layer

**`ESM2TrainConfig(mode="likelihoods")`** uses the bare backbone with no linear head. With the default `freeze_backbone=True`, `predict()` returns zero-shot pseudo-log-likelihood scores and no training occurs. To enable supervised prediction instead, use `mode="linear_head"` (the default) — covered in Section 5.

> **Contributors:** `ESM2Model.embed()` in `tools/alf_tools/models/esm2.py` tokenizes via `featurise()`, runs a forward pass with `output_hidden_states=True`, then pools the selected hidden layer.

In [ ]:
model_cfg = ESM2ModelConfig(
    model_id="facebook/esm2_t6_8M_UR50D",
    pooling="mean",
    repr_layer=-1,
)
train_cfg = ESM2TrainConfig(mode="likelihoods")  # frozen backbone -> zero-shot PLL

model = ESM2Model(name="esm2-gfp", model_config=model_cfg, train_config=train_cfg, device=device)
print(f"Loaded: {model.model_config.model_id}")
print(f"Embedding dim: {model.esm_model.config.hidden_size}")
print(f"Max sequence length: {model.max_length}")

## 3. Extract Sequence Embeddings

`embed()` runs a batched forward pass and returns a `(N, hidden_dim)` numpy array — one 320-dimensional vector per sequence.

Note: `featurise()` only tokenizes (returns `{"input_ids", "attention_mask"}`); the forward pass and pooling live entirely in `embed()`.

In [ ]:
X = model.embed(embed_candidates)  # shape (50, 320)
print(f"Embedding shape: {X.shape}")

## 3b. Zero-Shot PLL Scoring

`predict()` on a zero-shot model (no linear head) scores sequences via **pseudo-log-likelihood (PLL)**.
It masks one residue at a time and accumulates `log P(token_i | all other tokens)` from the frozen backbone —
no training required. Higher PLL = sequence is more probable under the model.

`variances` is always `None` for ESM-2 (no uncertainty estimate).

> **Slow on CPU.** Because PLL masks one residue at a time, scoring a single sequence costs one
forward pass *per residue*. On CPU this step can take a while, so only three sequences are scored
below. On a GPU it is far faster.

In [ ]:
pll_preds = model.predict(embed_candidates[:3])
print(f"Zero-shot PLL scores: {pll_preds.means}")
print(f"Variances           : {pll_preds.variances}")  # None

## 4. UMAP of Frozen Embeddings

We project 320-dimensional embeddings to 2D with UMAP and colour each point by fluorescence. If ESM-2 has captured meaningful structure in GFP sequence space, high-brightness variants should cluster rather than scatter uniformly.

In [ ]:
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
coords = reducer.fit_transform(X)

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(coords[:, 0], coords[:, 1], c=embed_labels, cmap="viridis", s=15, alpha=0.8)
plt.colorbar(sc, ax=ax, label="Median Brightness")
ax.set_title("UMAP of ESM-2 embeddings (frozen) — GFP sequences")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

## 5. Train a Linear Regression Head

For supervised fitness prediction, set `mode="linear_head"` in `ESM2TrainConfig`. This freezes the ESM-2 backbone and trains a small linear head on top of pooled sequence embeddings using labelled fitness data.

We split our 50 sequences into 40 train / 10 val and train for 2 epochs — enough to see the loss decrease on CPU in under 2 minutes.

> `loss_fn="mse"` with `output_dim=1` configures a regression head. For classification, use `loss_fn="cross_entropy"` and set `output_dim` to the number of classes.

In [ ]:
n_train = 40
train_data = LabelledCandidates(
    candidates=finetune_candidates[:n_train],
    labels=finetune_labels[:n_train],
)
val_data = LabelledCandidates(
    candidates=finetune_candidates[n_train:],
    labels=finetune_labels[n_train:],
)

Here we build an `ESM2TrainConfig` with `mode="linear_head"`, instantiate a fresh `ESM2Model`, and call `train()` on the labelled split. With `freeze_backbone` left at its default, only the linear head learns while the backbone stays frozen. `get_training_summary_metrics()` then returns the final train (and validation) losses from this run.

In [ ]:
train_cfg_reg = ESM2TrainConfig(
    mode="linear_head",
    loss_fn="mse",
    output_dim=1,
    num_epochs=2,
    batch_size=4,
    learning_rate=1e-3,
)
model_reg = ESM2Model(
    name="esm2-gfp-reg", model_config=model_cfg, train_config=train_cfg_reg, device=device
)
model_reg.train(train_data, val_data)

metrics_reg = model_reg.get_training_summary_metrics()
print("Linear head summary metrics:", metrics_reg)

`get_epoch_metrics()` returns one `SurrogateEpochMetrics` record per logged epoch, exposing `train_loss` and (when validation data is supplied) `val_loss`. We print them here to watch the loss move epoch by epoch rather than just the final summary.

In [ ]:
epoch_metrics = model_reg.get_epoch_metrics()
print(f"Recorded {len(epoch_metrics)} epoch checkpoints")
for m in epoch_metrics:
    line = f"  Epoch {m.epoch + 1}: train_loss={m.train_loss:.4f}"
    if m.val_loss is not None:
        line += f", val_loss={m.val_loss:.4f}"
    print(line)

To sanity-check the trained head we predict on the held-out validation candidates and plot predicted against true brightness. Points hugging the dashed `y = x` line would indicate accurate predictions; the interpretation below explains what to make of the scatter you actually see.

In [ ]:
val_preds = model_reg.predict(finetune_candidates[n_train:])

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(finetune_labels[n_train:], val_preds.means, alpha=0.8, s=40)
ax.axline((0, 0), slope=1, color="gray", linestyle="--", linewidth=1, label="y = x")
ax.set_xlabel("True Brightness")
ax.set_ylabel("Predicted Brightness")
ax.set_title("Validation predictions — linear head on ESM-2")
ax.legend()
plt.tight_layout()
plt.show()

> **These predictions look off — by design.** The points do not track the `y = x` line, and that
is expected here. The head is deliberately undertrained: only 50 sequences (40 train / 10 val) and
2 epochs are used to keep the tutorial fast on CPU. With so little data and so few steps, the linear
head cannot fit the brightness signal, so the scatter is essentially flat or noisy. This cell is
illustrative of the `mode="linear_head"` API and workflow, **not** a real fitness-prediction
benchmark. For meaningful accuracy you would train on far more sequences for many more epochs (and
ideally on translated amino-acid sequences rather than the raw nucleotides used here).

In [ ]:
metrics_df = pd.DataFrame([metrics_reg], index=["linear head"])
display(metrics_df)

## 6. Predict with the Trained Model

`model_reg.predict()` returns regression values from the trained linear head, wrapped in a `Predictions` object whose `.means` is a `(N,)` numpy array. `.variances` is always `None` for ESM-2.

In [ ]:
test_candidates = data.candidates[200:210]

reg_preds = model_reg.predict(test_candidates)
print(f"Predictions shape  : {reg_preds.means.shape}")  # (10,)
print(f"First 3 predictions: {reg_preds.means[:3]}")
print(f"Variances          : {reg_preds.variances}")  # None

## 7. Fine-Tune the Backbone with MLM

The third mode unfreezes the full ESM-2 backbone and fine-tunes it with **masked language modelling (MLM)** — the same objective ESM-2 was pre-trained on. A random fraction of residues (`mask_probability`) is corrupted following the BERT-style 80/10/10 `mask_splitting` rule, and the model learns to recover the originals on *your* sequences, adapting the backbone's likelihood landscape to a specific protein family.

MLM is unsupervised: it **ignores labels**, but `train()` still requires `LabelledCandidates`, so the brightness labels from earlier serve as harmless placeholders. After fine-tuning, `predict()` returns PLL scores (as in Section 3b) — now reflecting the adapted backbone.

> Each epoch records a token-weighted `train_loss`, plus `perplexity` and `token_accuracy` over the masked positions in `additional_metrics`. Validation masking is seeded from `model_config.seed`, so val metrics are comparable across epochs.

In [ ]:
train_cfg_mlm = ESM2TrainConfig(
    mode="likelihoods",
    freeze_backbone=False,
    loss_fn="mlm",
    mask_probability=0.15,
    num_epochs=2,
    batch_size=4,
    learning_rate=1e-4,
)
model_mlm = ESM2Model(
    name="esm2-gfp-mlm", model_config=model_cfg, train_config=train_cfg_mlm, device=device
)
# MLM ignores labels; train_data/val_data supply them only to satisfy LabelledCandidates.
model_mlm.train(train_data, val_data)

print("MLM summary metrics:", model_mlm.get_training_summary_metrics())

For the MLM run, `get_epoch_metrics()` carries the extra masked-language-modelling signals in each record's `additional_metrics`: `train_perplexity` and `train_token_accuracy` over the masked positions. We pull those out alongside the losses to see how well the backbone recovers masked residues as fine-tuning proceeds.

In [ ]:
mlm_epoch_metrics = model_mlm.get_epoch_metrics()
for m in mlm_epoch_metrics:
    train_ppl = m.additional_metrics.get("train_perplexity")
    train_acc = m.additional_metrics.get("train_token_accuracy")
    line = (
        f"Epoch {m.epoch + 1}: train_loss={m.train_loss:.4f}, "
        f"perplexity={train_ppl:.3f}, token_acc={train_acc:.3f}"
    )
    if m.val_loss is not None:
        line += f", val_loss={m.val_loss:.4f}"
    print(line)

Finally we score the same three sequences with the frozen zero-shot backbone (Section 2) and the MLM fine-tuned backbone, then compare their PLL values. Because fine-tuning adapts the model's likelihood landscape to these GFP sequences, the after scores should shift relative to the before scores.

In [ ]:
compare = embed_candidates[:3]
before = model.predict(compare).means  # frozen zero-shot backbone (Section 2)
after = model_mlm.predict(compare).means  # MLM fine-tuned backbone

print("PLL before vs after MLM fine-tuning:")
for i, (b, a) in enumerate(zip(before, after)):
    print(f"  seq {i}: {b:.3f} -> {a:.3f}")